# Dense structural channels for the coarse stage (iteration 29)

**Why.** Above ~800 m single dark spots are about one pixel and LoFTR finds nothing, but broad tracks, streaks and regional texture remain (Codex review 2026-11-18, our iterations 24-28). Does a coarse search on *larger-scale* structure find the place, and does agreement between independent channels give a safe answer rule on all three data sets?

**Channels** (all built per 4x4-pixel cell, the same way for the map window and the photo canvas, then correlated with the same masked ZNCC over the 5.5 km disc as the edge search):
- `edge`: the current edge-orientation search (baseline, from the store).
- `orient`: edge orientation at a coarser scale (gradient of the cell-mean image blurred 1.5 cells; 8 directions).
- `texture`: two planes, ridge strength (Hessian, 1.5 cells) and local contrast (std of the pixels inside a cell, blurred) = a density of small dark/bright structures even when single spots are not resolved.
- `dog`: band-pass (difference of Gaussians 1 and 3 cells) of the cell-mean image, signed.
- `fuse_eo` = z(edge) + z(orient); `fuse_all` = sum of the z of all four.

**Queries.** 3 per photo: headings -5, 0, +5 at scale 1.0 (the 6-query set is too slow for 5 searches each). Hint jittered 5 km as in the real test. Per query and channel: best peak, its z over the disc, error in sigma, truth in top 5, and the **decoy** peak = best peak after blanking the true place (radius max(15 sigma, half a canvas)).

**Wrong-map control (the review's idea, done inside the same map).** With the true place removed, any agreement between the edge decoy peak and a channel decoy peak is a false agreement. The agreement gate must not pass on decoys.

**Kill rule (written before the run).** A channel is useful only if on all three sets the answer rule "edge answer and channel answer within 1 sigma of each other" gives 0 wrong answers (beyond 8 sigma) AND answers at least 40 % of the photos above 800 m on the new set AND the false-agreement rate on decoys is 0 in the photos tested. If wrong answers appear on one set the rule does not transfer.

Env: `CH_HEADLESS, CH_DATASET, CH_STORE, CH_LEVELS, CH_TAG, CH_FIRST, CH_LIMIT, CH_JITTER, CH_OUT, CH_GEOMETRY`.

In [ ]:
import os, sys, math
import numpy as np, pandas as pd, torch, cv2
HEADLESS = os.environ.get("CH_HEADLESS", "../../../../Main_Pipeline_Headless")
sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [int(v) for v in os.environ.get("CH_LEVELS", "3,4,5,6,7").split(",")]
for item in os.environ.get("CH_GEOMETRY", "").split(","):
    if item:
        key, value = item.split("="); settings.GEOMETRY[key] = (value == "True") if value in ("True", "False") else float(value)
from geoloc import photo, queries, scoring
from geoloc.cli import read_rows, move_hint
from geoloc.coarse_search import DEVICE, score_map_for_query, pick_peaks, window_transforms, zncc_everywhere, fast_length
from geoloc.levels import level_for_height, level_gsd
from geoloc.map_store import MapStore
CELL = settings.CELL
DATASET = os.environ["CH_DATASET"]; STORE = os.environ["CH_STORE"]
TAG = os.environ.get("CH_TAG", "run"); FIRST = int(os.environ.get("CH_FIRST", 0)); LIMIT = int(os.environ.get("CH_LIMIT", 0))
JITTER = float(os.environ.get("CH_JITTER", 5000)); OUT = os.environ.get("CH_OUT", "results/coarse_channels"); os.makedirs(OUT, exist_ok=True)
store = MapStore(STORE); rows = read_rows(DATASET)[FIRST:]
if LIMIT: rows = rows[:LIMIT]
query_list = [q for q in queries.query_list() if abs(q["scale_factor"] - 1.0) < 1e-6]
print(DEVICE, len(rows), "photos,", len(query_list), "queries, levels", settings.LEVELS)

In [ ]:
# ---- channel features, per 4x4 cell, identical code for the map window and the canvas ----
def cell_stats_window(level_image, top, left, cells_wide):
    # mean, std and has-map of every cell of the pixel window (zeros outside the map), in strips to save memory
    mean = np.zeros((cells_wide, cells_wide), np.float32); std = np.zeros_like(mean); has = np.zeros((cells_wide, cells_wide), bool)
    height, width = level_image.shape; r0 = top * CELL; c0 = left * CELL; strip = 256
    for s in range(0, cells_wide, strip):
        n = min(strip, cells_wide - s); pr0 = r0 + s * CELL; pr1 = pr0 + n * CELL
        piece = np.zeros((n * CELL, cells_wide * CELL), np.float32)
        a0, a1 = max(pr0, 0), min(pr1, height); b0, b1 = max(c0, 0), min(c0 + cells_wide * CELL, width)
        if a1 > a0 and b1 > b0:
            piece[a0 - pr0:a1 - pr0, b0 - c0:b1 - c0] = np.asarray(level_image[a0:a1, b0:b1], dtype=np.float32) / 255.0
        blocks = piece.reshape(n, CELL, cells_wide, CELL)
        mean[s:s + n] = blocks.mean(axis=(1, 3)); std[s:s + n] = blocks.std(axis=(1, 3)); has[s:s + n] = blocks.min(axis=(1, 3)) > 0
    has = cv2.erode(has.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    return mean, std, has

def cell_stats_canvas(canvas, valid):
    side = canvas.shape[0] // CELL
    blocks = canvas[:side * CELL, :side * CELL].astype(np.float32).reshape(side, CELL, side, CELL)
    safe = cv2.erode(valid.astype(np.uint8), np.ones((7, 7), np.uint8))
    has = safe.reshape(side, CELL, side, CELL).mean(axis=(1, 3)) >= 0.8
    return blocks.mean(axis=(1, 3)), blocks.std(axis=(1, 3)), has

def blur(x, sigma): return cv2.GaussianBlur(x.astype(np.float32), (0, 0), sigma)

def channel_planes(mean, std, has):
    m = mean.copy()
    if has.any(): m[~has] = mean[has].mean()
    out = {}
    out["dog"] = (blur(m, 1.0) - blur(m, 3.0))[None]
    s1 = blur(m, 1.5)
    hxx = cv2.Sobel(s1, cv2.CV_32F, 2, 0, ksize=3); hyy = cv2.Sobel(s1, cv2.CV_32F, 0, 2, ksize=3); hxy = cv2.Sobel(s1, cv2.CV_32F, 1, 1, ksize=3)
    trace = (hxx + hyy) / 2; spread = np.sqrt(((hxx - hyy) / 2) ** 2 + hxy ** 2); l1 = np.abs(trace + spread); l2 = np.abs(trace - spread)
    ridge = np.clip(np.maximum(l1, l2) - np.minimum(l1, l2), 0, None)
    out["texture"] = np.stack([ridge, blur(std, 1.0)])
    gx = cv2.Sobel(s1, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(s1, cv2.CV_32F, 0, 1, ksize=3)
    strength = np.hypot(gx, gy); angle = np.arctan2(gy, gx) % np.pi; bin_id = np.minimum((angle / np.pi * settings.N_BINS).astype(int), settings.N_BINS - 1)
    orient = np.stack([blur(strength * (bin_id == k), 0.7) for k in range(settings.N_BINS)])
    length = np.sqrt((orient ** 2).sum(axis=0)); ref = np.percentile(length[has], 30) if has.any() else 1.0
    out["orient"] = orient / (length + ref + 1e-6)
    for name in ("dog", "texture"):
        for k in range(len(out[name])):
            scale = np.percentile(np.abs(out[name][k][has]), 70) if has.any() else 1.0
            out[name][k] = out[name][k] / (scale + 1e-6)
    for name in out: out[name][:, ~has] = 0
    return {name: np.ascontiguousarray(f, dtype=np.float32) for name, f in out.items()}

In [ ]:
# ---- search geometry (same formulas as coarse_search.score_map_for_query) and the channel search ----
def window_geometry(prior, side, level):
    gsd = level_gsd(level); reach = settings.DISC_RADIUS / gsd + side * CELL / 2
    cells_wide = int(math.ceil(2 * reach / CELL))
    return dict(gsd=gsd, cells_wide=cells_wide, top=int(round((prior[1] / gsd - reach) / CELL)), left=int(round((prior[0] / gsd - reach) / CELL)))

def channel_search(window, shared, size, kernel, cells, geo, side, level, prior):
    zncc = zncc_everywhere(window, shared, size, torch.from_numpy(kernel).to(DEVICE), torch.from_numpy(cells.astype(np.float32)).to(DEVICE))
    row_ids = torch.arange(zncc.shape[0], device=DEVICE); col_ids = torch.arange(zncc.shape[1], device=DEVICE)
    y_m = ((geo["top"] + row_ids) * CELL + side * CELL / 2) * geo["gsd"]; x_m = ((geo["left"] + col_ids) * CELL + side * CELL / 2) * geo["gsd"]
    inside = (x_m[None, :] - float(prior[0])) ** 2 + (y_m[:, None] - float(prior[1])) ** 2 <= settings.DISC_RADIUS ** 2
    score_map = torch.where((zncc > -1) & inside, zncc, torch.full_like(zncc, -1.0))
    return score_map, dict(level=level, side=side, x_m=x_m, y_m=y_m, inside=inside)

def disc_z_map(score_map, geometry):
    usable = (score_map > -1) & geometry["inside"]; v = score_map[usable]
    return torch.where(usable, (score_map - v.mean()) / v.std(), torch.zeros_like(score_map)), usable

def fuse(maps, geometry):
    total = None; usable_all = None
    for m in maps:
        z, usable = disc_z_map(m, geometry)
        total = z if total is None else total + z; usable_all = usable if usable_all is None else usable_all & usable
    return torch.where(usable_all, total + 100.0, torch.full_like(total, -1.0))   # +100 keeps every usable value above the -1 'unusable' marker

def peak_record(score_map, geometry, truth, sigma, level):
    usable = (score_map > -1) & geometry["inside"]; v = score_map[usable]
    peaks = pick_peaks(score_map, geometry, 5)
    errs = np.hypot(peaks[:, 1] - truth[0], peaks[:, 2] - truth[1]) / sigma
    z = float((peaks[0, 0] - v.mean()) / v.std())
    # decoy: blank the true place and take the best peak that is left
    radius = max(15 * sigma, 0.5 * settings.TILE * level_gsd(level))
    far = ((geometry["x_m"][None, :] - truth[0]) ** 2 + (geometry["y_m"][:, None] - truth[1]) ** 2) > radius ** 2
    decoy_map = torch.where(far, score_map, torch.full_like(score_map, -1.0)); dpeak = pick_peaks(decoy_map, geometry, 1)[0]
    dv = decoy_map[(decoy_map > -1) & geometry["inside"]]
    dz = float((dpeak[0] - dv.mean()) / dv.std()) if len(dv) > 10 else float("nan")
    return dict(x=peaks[0, 1], y=peaks[0, 2], z=z, err=errs[0], top5=float(errs.min()), dx=dpeak[1], dy=dpeak[2], dz=dz)

In [ ]:
records = []; GROUPS = ["dog", "texture", "orient"]
for index, meta in enumerate(rows):
    meta = move_hint(meta, index + FIRST, JITTER, 0) if JITTER > 0 else meta
    camera = photo.read_camera(meta); pose = photo.read_pose(meta)
    prior = store.gps_to_metres(float(meta["lat_sc"]), float(meta["lon_sc"])); truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"]))
    sigma = scoring.sigma_m(pose["height"], camera["fx"]); level = level_for_height(pose["height"], camera)
    picture = photo.prepare_photo(os.path.join(DATASET, "images", meta["filename"]), pose, camera, level)
    side = settings.TILE // CELL; geo = window_geometry(prior, side, level)
    try:
        mean, std, has = cell_stats_window(store.levels[level], geo["top"], geo["left"], geo["cells_wide"])
        planes = channel_planes(mean, std, has)
        size = (fast_length(geo["cells_wide"]), fast_length(geo["cells_wide"]))
        windows = {}
        for name in GROUPS:
            w = torch.from_numpy(planes[name]).to(DEVICE); windows[name] = (w, window_transforms(w, size))
        edge_cache = {}
        for q, query in enumerate(query_list):
            canvas, valid = photo.draw_canvas(picture, pose, camera, level, query["heading"], query["scale_factor"])
            maps = {}
            maps["edge"], geometry = score_map_for_query(canvas, valid, level, prior, store.features, edge_cache)
            cmean, cstd, chas = cell_stats_canvas(canvas, valid); cplanes = channel_planes(cmean, cstd, chas)
            for name in GROUPS:
                maps[name], _ = channel_search(windows[name][0], windows[name][1], size, cplanes[name], chas, geo, side, level, prior)
            maps["fuse_eo"] = fuse([maps["edge"], maps["orient"]], geometry)
            maps["fuse_all"] = fuse([maps["edge"]] + [maps[n] for n in GROUPS], geometry)
            for name, score_map in maps.items():
                r = peak_record(score_map, geometry, truth, sigma, level)
                records.append(dict(photo=index + FIRST, name=meta["filename"], q=q, height=pose["height"], sigma_m=sigma, level=level, channel=name, **r))
    except (ValueError, RuntimeError) as error:
        print("photo", index + FIRST, "skipped:", error)
    if index % 10 == 0:
        print(index + 1, "/", len(rows), flush=True); pd.DataFrame(records).to_csv(os.path.join(OUT, "channels_%s.csv" % TAG), index=False)
table = pd.DataFrame(records); table.to_csv(os.path.join(OUT, "channels_%s.csv" % TAG), index=False); print(len(table), "rows saved")

## Quick look: which channel finds the place, by height band (best-z query per photo and channel)

In [ ]:
def points(err): return np.where(err <= 1, 10, np.where(err <= 2, 6, np.where(err <= 4, 3, np.where(err <= 8, 0, -20))))
best = table.loc[table.groupby(["photo", "channel"]).z.idxmax()]
best["band"] = pd.cut(best.height, [0, 500, 800, 1300, 2200])
print(best.groupby(["band", "channel"], observed=True).agg(n=("err", "size"), right2=("err", lambda e: (e <= 2).mean()), wrong=("err", lambda e: (e > 8).mean()),
      top5=("top5", lambda t: (t <= 2).mean()), pts=("err", lambda e: points(e.values).mean())).round(2).to_string())